In [1]:
import torch

from laya.decision import (
    TinyDecisionModel,
    TinyTransformerEncoder,
)

In [2]:
torch.manual_seed(42)
tokens = [
    "[PAD]",
    "[STATE]",
    "[QUESTION]",
    "[OPTIONS]",
    "[MASK]",
    "the",
    "customer",
    "was",
    "charged",
    "twice",
    "which",
    "department",
    "should",
    "handle",
    "this",
    "billing",
    "technical",
    "sales",
    "wants",
    "support",
]

token_to_id = {token: idx for idx, token in enumerate(tokens)}

mask_token_id = token_to_id["[MASK]"]

In [3]:
sequence = [
    "[STATE]",
    "the",
    "customer",
    "was",
    "charged",
    "twice",
    "[QUESTION]",
    "which",
    "department",
    "should",
    "handle",
    "this",
    "[OPTIONS]",
    "[MASK]",
    "billing",
    "[MASK]",
    "technical",
    "[MASK]",
    "sales",
]

token_ids = torch.tensor([[token_to_id[token] for token in sequence]])

token_ids, token_ids.shape

(tensor([[ 1,  5,  6,  7,  8,  9,  2, 10, 11, 12, 13, 14,  3,  4, 15,  4, 16,  4,
          17]]),
 torch.Size([1, 19]))

In [4]:
option_mask = token_ids == mask_token_id

option_mask, option_mask.sum()

(tensor([[False, False, False, False, False, False, False, False, False, False,
          False, False, False,  True, False,  True, False,  True, False]]),
 tensor(3))

In [5]:
encoder = TinyTransformerEncoder(
    vocab_size=len(tokens),
    max_seq_len=32,
    d_model=32,
    n_heads=4,
    d_ff=64,
    n_layers=2,
    causal=False,
)

model = TinyDecisionModel(
    encoder=encoder,
    d_model=32,
    decision_hidden_dim=64,
)

model

TinyDecisionModel(
  (encoder): TinyTransformerEncoder(
    (token_embedding): Embedding(20, 32)
    (position_embedding): Embedding(32, 32)
    (layers): ModuleList(
      (0-1): 2 x TransformerBlock(
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (attention): MultiHeadSelfAttention(
          (q_proj): Linear(in_features=32, out_features=32, bias=False)
          (k_proj): Linear(in_features=32, out_features=32, bias=False)
          (v_proj): Linear(in_features=32, out_features=32, bias=False)
          (out_proj): Linear(in_features=32, out_features=32, bias=False)
        )
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (ff): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=32, out_features=64, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=64, out_features=32, bias=True)
          )
        )
      )
    )
    (norm): LayerNorm

In [6]:
model.eval()

with torch.no_grad():
    logits, probabilities = model(
        token_ids=token_ids,
        option_mask=option_mask,
    )

logits, logits.shape, probabilities, probabilities.sum(dim=-1)

(tensor([[-0.1427,  0.1765, -0.4025]]),
 torch.Size([1, 3]),
 tensor([[0.3177, 0.4372, 0.2450]]),
 tensor([1.]))

In [7]:
options = [
    "billing",
    "technical",
    "sales",
]

for option, probability in zip(
    options,
    probabilities[0],
):
    print(f"{option:10s} " f"{probability.item():.4f}")

billing    0.3177
technical  0.4372
sales      0.2450


In [8]:
with torch.no_grad():
    hidden_states = encoder(token_ids)

hidden_states, hidden_states.shape

(tensor([[[-1.3220e+00, -3.5114e-01,  1.3841e+00,  1.8367e+00, -1.1880e-01,
           -4.0338e-01, -1.9319e-01, -5.0969e-02, -7.5048e-01,  1.2043e+00,
           -1.2776e+00,  1.0142e+00, -6.0425e-01,  1.6448e+00, -1.4137e+00,
           -4.0914e-01, -1.0778e+00, -2.9490e-02,  1.6326e-01, -9.2937e-01,
           -1.5382e-01,  1.6102e+00, -6.0223e-01, -4.1317e-01, -1.5136e+00,
            5.5141e-01, -4.3622e-01,  9.2467e-01, -5.7272e-01,  2.3394e+00,
           -2.0813e-01,  1.5818e-01],
          [ 3.3574e-01, -8.9102e-01,  6.2708e-02, -1.0717e+00, -2.5438e-01,
            1.9273e-01,  2.4805e-01, -9.4973e-01, -2.5723e+00,  2.7625e-01,
            1.4164e+00, -1.1909e+00, -4.4005e-01,  6.0678e-02,  2.9745e-01,
            1.8714e+00, -3.0880e-02,  7.0363e-01, -6.6793e-01,  1.5026e+00,
           -4.3130e-01, -1.2195e+00, -6.9667e-01,  7.9696e-01, -7.7425e-01,
           -8.4583e-01, -1.3898e-01,  8.2536e-03, -3.3054e-01,  1.6996e+00,
            1.6642e+00,  1.3691e+00],
          [ 

In [9]:
option_states = hidden_states[option_mask]

option_states, option_states.shape

(tensor([[ 1.5176,  0.4024, -1.0299, -1.0926,  0.0574,  1.7805,  1.2113,  0.7406,
           1.0665, -0.6771, -0.7996,  0.2851, -0.8131, -0.0764, -1.6992, -1.5942,
          -1.0075,  0.1264,  0.9941,  1.6270, -0.3628,  0.3756, -0.3623,  0.2429,
           0.5923,  1.1525,  0.2247,  1.1156, -1.9704, -1.1290, -0.5356, -0.3628],
         [ 1.1657,  0.0831, -0.6898, -0.5300, -1.5716,  0.9604,  1.1207, -0.5747,
          -0.0879, -0.3487, -1.2333,  0.2429,  0.1195,  0.6581, -0.7840,  2.1773,
          -1.6273,  0.4115,  0.4404,  0.8856,  0.0990, -0.7223,  0.2048,  0.3728,
          -0.4144,  1.9554, -0.0595,  0.8680, -2.7297, -0.7685,  0.1046,  0.2719],
         [ 1.8336,  0.2845, -1.0421, -0.5764, -0.6553,  0.2203, -0.4619, -1.1578,
          -0.2010,  0.0319, -1.0263,  0.3492,  0.0938,  1.5261, -0.5255,  1.1404,
          -0.8314,  0.8550,  0.8879,  1.7315, -0.0300, -0.1039, -0.6778,  0.6305,
          -0.6926, -0.4785, -0.7510,  2.2441, -2.6196, -0.6663,  0.5070,  0.1617]]),
 torch.Size

In [ ]:
billing_sequence = [
    "[STATE]",
    "the",
    "customer",
    "was",
    "charged",
    "twice",
    "[QUESTION]",
    "which",
    "department",
    "should",
    "handle",
    "this",
    "[OPTIONS]",
    "[MASK]",
    "billing",
    "[MASK]",
    "technical",
    "[MASK]",
    "sales",
]

technical_sequence = [
    "[STATE]",
    "the",
    "customer",
    "wants",
    "technical",
    "support",
    "[QUESTION]",
    "which",
    "department",
    "should",
    "handle",
    "this",
    "[OPTIONS]",
    "[MASK]",
    "billing",
    "[MASK]",
    "technical",
    "[MASK]",
    "sales",
]

billing_ids = torch.tensor([[token_to_id[token] for token in billing_sequence]])

technical_ids = torch.tensor([[token_to_id[token] for token in technical_sequence]])

billing_mask = billing_ids == mask_token_id
technical_mask = technical_ids == mask_token_id


with torch.no_grad():
    billing_hidden = encoder(billing_ids)
    technical_hidden = encoder(technical_ids)

billing_option_states = billing_hidden[billing_mask]
technical_option_states = technical_hidden[technical_mask]

print(billing_option_states, technical_option_states)

difference = (billing_option_states[0] - technical_option_states[0]).abs().mean()

difference

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True]])
tensor([[ 1.5176,  0.4024, -1.0299, -1.0926,  0.0574,  1.7805,  1.2113,  0.7406,
          1.0665, -0.6771, -0.7996,  0.2851, -0.8131, -0.0764, -1.6992, -1.5942,
         -1.0075,  0.1264,  0.9941,  1.6270, -0.3628,  0.3756, -0.3623,  0.2429,
          0.5923,  1.1525,  0.2247,  1.1156, -1.9704, -1.1290, -0.5356, -0.3628],
        [ 1.1657,  0.0831, -0.6898, -0.5300, -1.5716,  0.9604,  1.1207, -0.5747,
         -0.0879, -0.3487, -1.2333,  0.2429,  0.1195,  0.6581, -0.7840,  2.1773,
         -1.6273,  0.4115,  0.4404,  0.8856,  0.0990, -0.7223,  0.2048,  0.3728,
         -0.4144,  1.9554, -0.0595,  0.8680, -2.7297, -0.7685,  0.1046,  0.2719],
        [ 1.8336,  0.2845, -1.0421, -0.5764, -0.6553,  0.2203, -0.4619, -1.1578,
         -0.2010,  0.0319, -1.0263,  0.3492,  0.0938,  1.5261, -0.5255,  1.1404,
         -0.8314,  0.8550,  0.8879,  1.7315, -0.0300, 

tensor(0.0224)

In [11]:
with torch.no_grad():
    _, billing_probs = model(
        billing_ids,
        billing_mask,
    )

    _, technical_probs = model(
        technical_ids,
        technical_mask,
    )


for option, p_a, p_b in zip(
    options,
    billing_probs[0],
    technical_probs[0],
):
    print(
        f"{option:10s} "
        f"billing_state={p_a.item():.4f} "
        f"technical_state={p_b.item():.4f}"
    )

billing    billing_state=0.3177 technical_state=0.3164
technical  billing_state=0.4372 technical_state=0.4391
sales      billing_state=0.2450 technical_state=0.2445
